# Paired comparison of the controllers (SI Section 6.8, Table S13)

All controllers visited the same $20$ control targets (k-medoids control workspace, SI Section 6.4), so they are compared target by target:
mean error of controller A and B, number of targets on which A is more accurate, and $p$-value of the two-sided Wilcoxon signed-rank test on the paired errors.

**Inputs** (repository): `../SINGLE/results/ControlWorkspace_*.npz` and `results/ControlWorkspace*.npz` (arrays `target` and `mocap`; multi-section poses are $[x, y, z]$ plus ZYX Euler angles).
**Output**: `results/paired_comparison.json` and the table below. Run from inside `MULTI/`.

In [1]:
import json
import numpy as np
from scipy.stats import wilcoxon
from scipy.spatial.transform import Rotation as R

SINGLE = "../SINGLE/results/"
MULTI = "results/"


def ang_dist_deg(a, b):
    """Geodesic distance [deg] between two sets of ZYX Euler orientations."""
    return np.degrees(np.linalg.norm((R.from_euler("zyx", b) * R.from_euler("zyx", a).inv()).as_rotvec(), axis=-1))


def errors(path):
    """Position error [mm] and, for 6D poses, orientation error [deg] on every target."""
    d = np.load(path)
    t, m = d["target"], d["mocap"]
    pos = np.linalg.norm(t[:, :3] - m[:, :3], axis=1)
    rot = ang_dist_deg(t[:, 3:], m[:, 3:]) if t.shape[1] == 6 else None
    return {"position": pos, "orientation": rot}

## Control results on the 20 targets

In [2]:
single = {
    "model-based closed loop": errors(SINGLE + "ControlWorkspace_jacobian.npz"),
    "data-driven closed loop": errors(SINGLE + "ControlWorkspace_datadriven.npz"),
    "model-based open loop": errors(SINGLE + "ControlWorkspace_OL_jacobian.npz"),
    "data-driven open loop": errors(SINGLE + "ControlWorkspace_OL_datadriven.npz"),
}
multi = {
    "model-based closed loop": errors(MULTI + "ControlWorkspaceJacobian.npz"),
    "data-driven closed loop": errors(MULTI + "ControlWorkspaceDatadriven.npz"),
    "model-based open loop": errors(MULTI + "ControlWorkspaceOpenLoopJacobian.npz"),
    "data-driven open loop": errors(MULTI + "ControlWorkspaceOpenLoopDatadriven.npz"),
    "model-based closed loop, 50 g": errors(MULTI + "ControlWorkspaceJacobian_W50g.npz"),
    "model-based open loop, 50 g": errors(MULTI + "ControlWorkspaceOpenLoopJacobian_W50g.npz"),
}

## Table S13

In [3]:
# (arm, quantity, A, B) in the order of Table S13
ROWS = [
    ("Single", "position", "model-based closed loop", "data-driven closed loop"),
    ("Single", "position", "model-based closed loop", "model-based open loop"),
    ("Single", "position", "data-driven closed loop", "data-driven open loop"),
    ("Multi", "position", "model-based closed loop", "data-driven closed loop"),
    ("Multi", "position", "model-based closed loop", "model-based open loop"),
    ("Multi", "position", "data-driven closed loop", "data-driven open loop"),
    ("Multi", "position", "model-based closed loop, 50 g", "model-based open loop, 50 g"),
    ("Multi", "position", "model-based closed loop", "model-based closed loop, 50 g"),
    ("Multi", "position", "model-based open loop", "model-based open loop, 50 g"),
    ("Multi", "orientation", "model-based closed loop", "data-driven closed loop"),
    ("Multi", "orientation", "model-based closed loop", "model-based open loop"),
    ("Multi", "orientation", "data-driven closed loop", "data-driven open loop"),
    ("Multi", "orientation", "model-based closed loop, 50 g", "model-based open loop, 50 g"),
]

table = []
print(f"{'Arm':6s} {'quantity':11s} {'A vs B':62s} {'mean A / B':>14s} {'A better':>9s} {'p':>8s}")
for arm, q, a, b in ROWS:
    D = single if arm == "Single" else multi
    ea, eb = D[a][q], D[b][q]
    p = float(wilcoxon(ea, eb, alternative="two-sided").pvalue)
    n_better = int((ea < eb).sum())
    table.append({"arm": arm, "quantity": q, "A": a, "B": b, "mean_A": float(ea.mean()), "mean_B": float(eb.mean()),
                  "A_better": n_better, "n_targets": len(ea), "p": p})
    p_txt = "< 0.001" if p < 0.001 else (f"{p:.3f}" if p < 0.01 else f"{p:.2f}")
    print(f"{arm:6s} {q:11s} {a + ' vs ' + b:62s} {ea.mean():6.1f} / {eb.mean():5.1f} {n_better:9d} {p_txt:>8s}")

with open("results/paired_comparison.json", "w") as f:
    json.dump(table, f, indent=1)
print("\nSaved results/paired_comparison.json")

Arm    quantity    A vs B                                                             mean A / B  A better        p
Single position    model-based closed loop vs data-driven closed loop                4.6 /   7.5        20  < 0.001
Single position    model-based closed loop vs model-based open loop                  4.6 /  15.3        19  < 0.001
Single position    data-driven closed loop vs data-driven open loop                  7.5 /  11.0        17  < 0.001
Multi  position    model-based closed loop vs data-driven closed loop               25.3 /  27.4        10     0.87
Multi  position    model-based closed loop vs model-based open loop                 25.3 /  31.8        13     0.11
Multi  position    data-driven closed loop vs data-driven open loop                 27.4 /  41.1        19  < 0.001
Multi  position    model-based closed loop, 50 g vs model-based open loop, 50 g     28.5 /  64.6        18  < 0.001
Multi  position    model-based closed loop vs model-based closed loop, 5